In [25]:
import sys
sys.path.append("../src")

import pandas as pd
from data_loader import load_ratings

In [26]:
ratings = load_ratings("../data/raw/ratings.csv")

# Exploratory Data Analysis
The objective of this EDA is to analyze whether the ratings dataset has enough ratings per user and movie

In [27]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [28]:
ratings.shape

(100836, 4)

In [29]:
ratings["userId"].nunique()

610

In [30]:
ratings["movieId"].nunique()

9724

In [31]:
ratings["rating"].describe()

count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64

In [34]:
ratings.groupby("userId").size().describe()

count     610.000000
mean      165.304918
std       269.480584
min        20.000000
25%        35.000000
50%        70.500000
75%       168.000000
max      2698.000000
dtype: float64

In [40]:
ratings.groupby("movieId").size().describe()

count    9724.000000
mean       10.369807
std        22.401005
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       329.000000
dtype: float64

In [55]:
ratings.groupby("userId").agg(
    total_reviews = ('userId', 'count'),
    average_rating = ('rating', 'mean')
).sort_values(by="total_reviews", ascending=False)

,total_reviews,average_rating
userId,,
414,2698,3.391957
599,2478,2.642050
474,2108,3.398956
448,1864,2.847371
274,1346,3.235884
...,...,...
442,20,1.275000
569,20,4.000000
320,20,3.525000


In [60]:
movies_ratings = ratings.groupby("movieId").agg(
    total_reviews = ('movieId', 'count'),
    avg_rating = ('rating', 'mean')
).sort_values(by="total_reviews", ascending=False)

movies_ratings

,total_reviews,avg_rating
movieId,,
356,329,4.164134
318,317,4.429022
296,307,4.197068
593,279,4.161290
2571,278,4.192446
...,...,...
4093,1,1.500000
4089,1,2.000000
58351,1,4.000000


In [ ]:
movies_ratings[movies_ratings["total_reviews"] == 1]["total_reviews"].count()

np.int64(3446)

The analyzed data has enough quality for a solid collaborative filtering approach, as there are at least 20 reviews per user. The only concern is that 3,446 movies have just one review, which means we don't have enough data to recommend them properly

## Sparsity of the matrix

In [62]:
num_users = ratings["userId"].nunique()
num_movies = ratings["movieId"].nunique()
num_ratings = ratings.shape[0]

total_possible_ratings = num_users * num_movies

sparsity = 1 - (num_ratings / total_possible_ratings)

sparsity

0.9830003169443864

The user-item matrix is highly sparse, which is expected in recommender systems because each user only rates a small subset of movies.

## User-movie matrix

In [66]:
user_item_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

user_item_matrix.head(15)

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,4.0,5.0,3.0,5.0,4.0,4.0,3.0,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
user_item_matrix.shape

(610, 9724)